In [ ]:
!module load bcftools

In [ ]:
!pip install pandas numpy biopython rpy2

In [ ]:
import subprocess
import pandas as pd
import os

In [ ]:
# Define the samples and their populations
SAN = ["KSP062", "KSP063", "KSP065", "KSP067", "KSP069", "KSP092", "KSP096", "KSP103", "KSP105", "KSP111", "KSP116", "KSP124", "KSP134", "KSP137", "KSP139", "KSP140", "KSP146", "KSP150", "KSP152", "KSP154", "KSP155", "KSP224", "KSP225", "KSP228"]
RHGn = ["HG02568", "HG02922", "HG03052", "HGDP_HGDP00927", "HGDP_HGDP01284", "SGDP_LP6005441-DNA_E07", "SGDP_LP6005441-DNA_F07", "SGDP_LP6005442-DNA_A02", "SGDP_LP6005442-DNA_A10", "SGDP_LP6005442-DNA_B02", "SGDP_LP6005442-DNA_B10", "SGDP_LP6005442-DNA_G10", "SGDP_LP6005442-DNA_G11", "SGDP_LP6005442-DNA_H10", "SGDP_SS6004475", "SGDP_SS6004470", "HGDP_DNK02", "NA19017", "PNP010", "PNP011", "PNP012", "PNP013", "PNP014", "PNP030", "PNP031"]
Current = ["KSP111"]

In [ ]:
# Combine all samples
all_samples = SAN + RHGn + Current 

# Create a mapping of sample to population
sample_population = {}
for sample in SAN:
    sample_population[sample] = 'SAN'
for sample in RHGn:
    sample_population[sample] = 'RHGn'
for sample in Current:
    sample_population[sample] = 'Current'

In [ ]:
# Create a comma-separated string of sample names
sample_list_str = ','.join(all_samples)

In [ ]:
bed_file = '/lisc/scratch/admixlab/aigerim/sstar-analysis-main/African/San21/KSP111/KSP111_overlap.bed'
regions_df = pd.read_csv(bed_file, sep='\s+', header=None)
print(regions_df.head())

In [ ]:
# Create a list of regions in the format: 'chrom:start-end'
regions = regions_df.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1).tolist()

In [ ]:
# Directory containing the VCF files for each chromosome
vcf_dir = '.'  # Replace with the directory containing the VCF files

# Base VCF file name pattern
#vcf_base_pattern = '25KS.48RHG.74comp.HCBP.{chr_num}.recalSNP99.9.recalINDEL99.0.vcf.gz'
vcf_base_pattern = '/lisc/scratch/admixlab/aigerim/shapeit5/African/phased/San_chr{chr_num}.phased.bcf'

# Directory to store extracted data
os.makedirs('extracted_data', exist_ok=True)

# Loop through chromosomes 1 to 22
for chr_num in range(1,23):
    # Format the chromosome-specific VCF file name (use just the number for the VCF filename)
    vcf_file = os.path.join(vcf_dir, vcf_base_pattern.format(chr_num=chr_num))
    
    # Check if the VCF file exists
    if not os.path.exists(vcf_file):
        print(f"VCF file {vcf_file} not found. Skipping chromosome {chr_num}.")
        continue
    
    print(f"Processing VCF file {vcf_file}...")
    
    # Filter regions that belong to the current chromosome (use 'chrN' for region filtering)
    chr_regions = [region for region in regions if region.startswith(f'chr{chr_num}:')]
    
    if not chr_regions:
        print(f"No regions found for chromosome {chr_num}. Skipping.")
        continue
    
    # Loop through each region in the current chromosome
    for region in chr_regions:
        print(f"Extracting data for region {region}...")
        
        # Construct the bcftools command
        command = f"bcftools view -m2 -M2 -v snps -r {region} {vcf_file} | bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT[\\t%GT]\\n'"
        
        # Run the command and capture the output
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"Error processing region {region}: {result.stderr}")
            continue
        
        # Process the output
        output = result.stdout
        
        # Save the output to a file
        filename = f'region_{region.replace(":", "_").replace("-", "_")}.txt'
        filepath = os.path.join('extracted_data', filename)
        
        with open(filepath, 'w') as f:
            f.write(output)
        
        print(f"Data for {region} saved to {filepath}")

print("Extraction completed.")


In [ ]:
# Get the list of sample names from the VCF file
vcf_samples_command = f"bcftools query -l {vcf_file}"
vcf_samples_result = subprocess.run(vcf_samples_command, shell=True, capture_output=True, text=True)
if vcf_samples_result.returncode != 0:
    print(f"Error getting sample names from VCF: {vcf_samples_result.stderr}")
    raise Exception("Failed to get sample names from VCF file.")

vcf_samples = vcf_samples_result.stdout.strip().split('\n')

# Prepare a dictionary to hold data for each region
region_data = {}

# List of extracted data files
data_files = [f for f in os.listdir('extracted_data') if f.startswith('region_') and f.endswith('.txt')]

for filename in data_files:
    filepath = os.path.join('extracted_data', filename)
    
    # Check if the file is empty
    if os.path.getsize(filepath) == 0:
        print(f"File {filename} is empty. Skipping.")
        continue
    
    # Read the data and print the first few lines for debugging
    try:
        data = pd.read_csv(filepath, sep='\t', header=None)
    except pd.errors.EmptyDataError:
        print(f"Error: File {filename} is empty or improperly formatted. Skipping.")
        continue

    # Assign column names
    num_cols = data.shape[1]
    
    # First four columns are CHROM, POS, REF, ALT, the rest are sample columns
    col_names = ['CHROM', 'POS', 'REF', 'ALT'] + vcf_samples
    if len(col_names) != num_cols:
        print(f"Column mismatch in {filename}: Expected {len(col_names)}, got {num_cols}. Skipping.")
        continue
    
    data.columns = col_names
    
    # Filter the data to include only the samples of interest
    data = data[['CHROM', 'POS', 'REF', 'ALT'] + all_samples]
    
    # Store the data in the dictionary
    region_data[filename] = data

print("Processing complete.")

In [ ]:
from collections import defaultdict
import random
import numpy as np
import pandas as pd

def safe_scalar(x):
    """
    If x is an iterable (list, numpy array, or pandas Series) and has at least one element, 
    return its first element; otherwise, return x unchanged.
    """
    if isinstance(x, (list, np.ndarray, pd.Series)):
        # If x has at least one element, return the first.
        if len(x) >= 1:
            return x[0]
    return x


def genotype_to_allele(row):
    """
    Convert genotype string for each sample in a row to a single allele.
    
    Rules:
      - If the genotype is missing (e.g. "./.", ".|.", or ".") return "N".
      - For homozygous calls ("0|0" or "1|1"), output "0" or "1" respectively.
      - For heterozygous calls ("0|1" or "1|0"), randomly choose either "0" or "1".
    """
    allele_list = []
    for sample in all_samples:
        gt = row[sample]
        gt = safe_scalar(gt)
        # If gt is a Series, extract the scalar value.
        if isinstance(gt, pd.Series):
            gt = gt.item()
        if gt in ['./.', '.|.', '.']:
            allele_list.append('N')
        elif gt == "0|0":
            allele_list.append("0")
        elif gt == "1|1":
            allele_list.append("1")
        elif gt in ["0|1", "1|0"]:
            allele_list.append(random.choice(["0", "1"]))
        else:
            allele_list.append('N')
    return allele_list

# Prepare a dictionary to hold haplotypes for each region.
# Assume 'region_data' is a dictionary where keys are region filenames
# and values are DataFrames that include columns: 'REF', 'ALT', and one column per sample.
# 'all_samples' is a list of sample names (columns in these DataFrames).
region_haplotypes = {}

for filename, data in region_data.items():
    print(f"Processing {filename}...")
    # Apply the conversion function to each row to obtain a list (one allele per sample)
    allele_matrix = data.apply(genotype_to_allele, axis=1)
    
    # Initialize a dictionary that will store one haplotype (consensus sequence) per sample
    haplotype = defaultdict(list)
    
    # Iterate over each row in the allele matrix and append the allele value for each sample.
    for idx in range(len(allele_matrix)):
        alleles = allele_matrix.iloc[idx]
        for i, sample in enumerate(all_samples):
            haplotype[sample].append(alleles[i])
    
    # Join the list of alleles into a single string for each sample.
    for sample in haplotype:
        haplotype[sample] = "".join(haplotype[sample])
    
    # Store the haplotype dictionary for the current region.
    region_haplotypes[filename] = haplotype


In [ ]:
import os
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations
from matplotlib.backends.backend_pdf import PdfPages

def hamming_distance(s1, s2):
    """Compute the Hamming distance between two equal-length sequences."""
    return sum(ch1 != ch2 for ch1, ch2 in zip(s1, s2))

def int_to_roman(num):
    """Convert an integer to a Roman numeral."""
    val = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]
    syms = ["M", "CM", "D", "CD", "C", "XC", "L", "XL", "X", "IX", "V", "IV", "I"]
    roman_num = ''
    i = 0
    while num > 0:
        for _ in range(num // val[i]):
            roman_num += syms[i]
            num -= val[i]
        i += 1
    return roman_num

def tree_layout(T, root, pos=None, visited=None, dx=1.0, dy=1.0):
    """
    Recursively compute positions for nodes in a tree T.
    Each child's x-coordinate = parent's x + (edge weight * dx).
    The y-coordinate is assigned to spread siblings using dy.
    Parameters:
      - T: the MST (a tree)
      - root: the starting node
      - pos: dictionary of positions (node: (x, y))
      - visited: set of visited nodes
      - dx: scaling factor for x (edge lengths)
      - dy: vertical spacing factor among siblings
    Returns:
      - pos: dictionary mapping node to (x, y)
    """
    if pos is None:
        pos = {root: (0, 0)}
    if visited is None:
        visited = set([root])
    children = [n for n in T.neighbors(root) if n not in visited]
    n = len(children)
    for i, child in enumerate(children):
        visited.add(child)
        # Get the edge weight from root to child and scale it with dx.
        weight = T.edges[root, child]['weight'] * dx
        # Compute vertical offset so siblings are spread out.
        y_offset = (i - (n - 1) / 2) * dy
        pos[child] = (pos[root][0] + weight, pos[root][1] + y_offset)
        tree_layout(T, child, pos, visited, dx, dy)
    return pos

# Determine label for the Current group:
if len(Current) == 1:
    current_label = Current[0]
else:
    current_label = "Current"

# Specify the output PDF filename.
pdf_filename = "KSP111_haplotype_networks_tree_layout.pdf"

with PdfPages(pdf_filename) as pdf:
    # Loop over all regions in your region_haplotypes dictionary.
    for region_filename, hap_dict in region_haplotypes.items():
        # -------------------------------------------------------------------
        # Extract a human-readable region string from the filename.
        # For example, "region_chr3_136330000_136950000.txt" becomes "chr3:136330000-136950000".
        base_name = os.path.splitext(os.path.basename(region_filename))[0]
        region_str = base_name
        if region_str.startswith("region_"):
            region_str = region_str.replace("region_", "", 1)
        parts = region_str.split('_')
        if len(parts) >= 3:
            region_str = f"{parts[0]}:{parts[1]}-{parts[2]}"
        # -------------------------------------------------------------------
        
        # Filter hap_dict to include samples in SAN, RHGn, or Current.
        filtered_hap_dict = {sample: seq for sample, seq in hap_dict.items()
                             if sample_population.get(sample) in ['SAN', 'RHGn', 'Current']}
        # Build a list of (sample, haplotype) tuples.
        hap_list = list(filtered_hap_dict.items())
        
        # Compute pairwise Hamming distances to build edges.
        edges = []
        for (s1, seq1), (s2, seq2) in combinations(hap_list, 2):
            dist = hamming_distance(seq1, seq2)
            edges.append((s1, s2, {'weight': dist}))
        
        # Create a graph with nodes as the sample names.
        G = nx.Graph()
        G.add_nodes_from(filtered_hap_dict.keys())
        G.add_edges_from(edges)
        
        # Compute the Minimum Spanning Tree (MST) of G.
        T = nx.minimum_spanning_tree(G, weight='weight')
        
        # Choose a root arbitrarily (first node).
        root = list(T.nodes())[0]
        # Use our custom tree_layout so that x-coordinates are cumulative edge weights.
        pos = tree_layout(T, root, dx=1.0, dy=0.5)
        
        # Create a label mapping: assign Roman numeral labels to nodes.
        node_list = list(T.nodes())
        label_map = {node: int_to_roman(i+1) for i, node in enumerate(node_list)}
        
        # Assign node colors based on population.
        node_colors = []
        for node in T.nodes():
            pop = sample_population.get(node)
            if pop == "SAN":
                node_colors.append("red")
            elif pop == "RHGn":
                node_colors.append("blue")
            elif pop == "Current":
                node_colors.append("yellow")
            else:
                node_colors.append("black")
        
        # Set up the figure.
        fig, ax = plt.subplots(figsize=(10, 8))
        nx.draw(T, pos, labels=label_map, with_labels=True,
                node_color=node_colors, node_size=1500,
                font_weight='bold', edge_color='gray', ax=ax)
        edge_labels = nx.get_edge_attributes(T, 'weight')
        nx.draw_networkx_edge_labels(T, pos, edge_labels=edge_labels, font_size=10, ax=ax)
        
        # Set the title showing the region and the group legend (including the Current sample's name).
        title_str = (f"Haplotype Network for Region: {region_str}\n"
                     f"SAN (red) vs. RHGn (blue) vs. {current_label} (yellow)")
        ax.set_title(title_str, fontsize=14)
        ax.axis('off')
        
        pdf.savefig(fig)
        plt.close(fig)

print(f"All region plots have been saved to {pdf_filename}")
